# No-Coordinate LLM Maze Experiment — Condition B + Smell

This notebook defines 15 solvable ASCII mazes and runs them through four LLMs
via Playwright automation of LibreChat.

**Condition B:** no coordinates, no overhead view, no visited markers, but the model
is told which direction it entered the current cell from. Backtracking is allowed.

**Smell:** each prompt also includes a crude as-the-crow-flies compass direction toward
the exit, quantized to one of eight directions (N, S, E, W, NE, NW, SE, SW).
This smell is *not* path-aware and can be misleading when the correct path requires
moving away from the end.

Baseline policies (random, non-backtracking, smell-greedy) are included for reference.

In [1]:

from dataclasses import dataclass
from collections import deque
from typing import Dict, Tuple, Set, List, Optional, Callable
import re
import random
import pandas as pd

Coord = Tuple[int, int]

DIRS = {
    "N": (-1, 0),
    "E": (0, 1),
    "S": (1, 0),
    "W": (0, -1),
}

DIR_WORDS = {
    "N": "north",
    "E": "east",
    "S": "south",
    "W": "west",
}

WORD_TO_DIR = {
    "n": "N",
    "north": "N",
    "e": "E",
    "east": "E",
    "s": "S",
    "south": "S",
    "w": "W",
    "west": "W",
}

OPPOSITE = {
    "N": "S",
    "E": "W",
    "S": "N",
    "W": "E",
}


@dataclass
class Maze:
    name: str
    size: int
    start: Coord
    end: Coord
    walls: Dict[Coord, Set[str]]
    shortest_path_length: Optional[int] = None


def parse_ascii_maze(
    name: str,
    ascii_maze: str,
    size: int,
    shortest_path_length: Optional[int] = None,
) -> Maze:
    """
    Parses an ASCII cell maze.

    Assumes:
    - ASCII dimensions are (2*size + 1) by (2*size + 1)
    - Maze cells are at ASCII positions (2*r + 1, 2*c + 1)
    - Walls are '#'
    - Passages are spaces, 'S', or 'E'

    Exterior openings are allowed visually, but the agent is not allowed to walk
    outside the grid during the experiment.
    """

    lines = ascii_maze.splitlines()
    expected_rows = 2 * size + 1
    expected_cols = 2 * size + 1

    if len(lines) != expected_rows:
        raise ValueError(f"{name}: expected {expected_rows} rows, got {len(lines)}")

    for i, line in enumerate(lines):
        if len(line) != expected_cols:
            raise ValueError(
                f"{name}: row {i} expected width {expected_cols}, got {len(line)}: {repr(line)}"
            )

    walls: Dict[Coord, Set[str]] = {}
    start = None
    end = None

    for r in range(size):
        for c in range(size):
            gr, gc = 2 * r + 1, 2 * c + 1
            ch = lines[gr][gc]

            if ch == "S":
                start = (r, c)
            elif ch == "E":
                end = (r, c)
            elif ch != " ":
                raise ValueError(f"{name}: invalid cell character at {(r, c)}: {repr(ch)}")

            cell_walls = set()
            wall_positions = {
                "N": (gr - 1, gc),
                "E": (gr, gc + 1),
                "S": (gr + 1, gc),
                "W": (gr, gc - 1),
            }

            for direction, (wr, wc) in wall_positions.items():
                if lines[wr][wc] == "#":
                    cell_walls.add(direction)

            walls[(r, c)] = cell_walls

    if start is None:
        raise ValueError(f"{name}: missing start cell S")

    if end is None:
        raise ValueError(f"{name}: missing end cell E")

    return Maze(
        name=name,
        size=size,
        start=start,
        end=end,
        walls=walls,
        shortest_path_length=shortest_path_length,
    )


def step_coord(coord: Coord, direction: str) -> Coord:
    dr, dc = DIRS[direction]
    r, c = coord
    return (r + dr, c + dc)


def in_bounds(maze: Maze, coord: Coord) -> bool:
    r, c = coord
    return 0 <= r < maze.size and 0 <= c < maze.size


def is_open(maze: Maze, coord: Coord, direction: str) -> bool:
    return direction not in maze.walls[coord]


def legal_moves(maze: Maze, coord: Coord) -> List[str]:
    """
    Legal internal moves. Exterior openings in the ASCII maze are ignored.
    The agent starts inside S and wins by reaching E.
    """

    moves = []
    for direction in ["N", "E", "S", "W"]:
        if not is_open(maze, coord, direction):
            continue

        nxt = step_coord(coord, direction)
        if in_bounds(maze, nxt):
            moves.append(direction)

    return moves


def apply_move(maze: Maze, coord: Coord, move: str) -> Tuple[Coord, bool, str]:
    move = move.strip().upper()

    if move not in DIRS:
        return coord, False, f"Invalid move token: {move!r}"

    if move not in legal_moves(maze, coord):
        return coord, False, f"Illegal move: {DIR_WORDS[move]} is blocked or exits the maze."

    return step_coord(coord, move), True, "ok"


def shortest_distance_from(maze: Maze, source: Coord, target: Coord) -> Optional[int]:
    q = deque([source])
    dist = {source: 0}

    while q:
        cur = q.popleft()

        if cur == target:
            return dist[cur]

        for move in legal_moves(maze, cur):
            nxt = step_coord(cur, move)

            if nxt not in dist:
                dist[nxt] = dist[cur] + 1
                q.append(nxt)

    return None


def validate_maze(maze: Maze, verbose: bool = True) -> bool:
    start_moves = legal_moves(maze, maze.start)
    end_dist = shortest_distance_from(maze, maze.start, maze.end)

    if not start_moves:
        raise ValueError(f"{maze.name}: start has no legal internal moves")

    if end_dist is None:
        raise ValueError(f"{maze.name}: end is unreachable from start")

    if maze.shortest_path_length is not None and maze.shortest_path_length != end_dist:
        raise ValueError(
            f"{maze.name}: stored shortest path length is {maze.shortest_path_length}, "
            f"but computed shortest path is {end_dist}"
        )

    if verbose:
        readable_start_moves = [DIR_WORDS[m] for m in start_moves]
        print(
            f"{maze.name}: valid | shortest path = {end_dist} | "
            f"start moves = {readable_start_moves}"
        )

    return True


def build_condition_b_prompt(
    maze: Maze,
    coord: Coord,
    recent_moves: List[str],
    max_recent_moves: int = 12,
) -> str:
    """
    Condition B:
    - No coordinates
    - No visited markers
    - Show local open directions
    - Show whether this is the start or end cell
    - Show the direction from which the agent entered the current cell
    - Allow backtracking
    """

    lines = []
    lines.append("You are navigating a maze.")
    lines.append("You cannot see the full maze.")
    lines.append("You only know which directions are open from your current cell.")
    lines.append("Your goal is to reach the end cell.")
    lines.append("You will be told when you are standing on the end cell.")
    lines.append("Reply with exactly one move: north, east, south, or west.")
    lines.append("Do not explain your reasoning.")
    lines.append("")

    if coord == maze.start:
        lines.append("This is the start cell.")

    if coord == maze.end:
        lines.append("This is the end cell. The maze is solved.")
        return "\n".join(lines)

    moves = legal_moves(maze, coord)
    readable_moves = [DIR_WORDS[m] for m in moves]
    lines.append(f"Available moves: {', '.join(readable_moves)}.")

    if recent_moves:
        previous_move = recent_moves[-1]
        came_from = OPPOSITE[previous_move]
        lines.append(f"You entered this cell from the {DIR_WORDS[came_from]}.")

        if max_recent_moves:
            readable_history = [DIR_WORDS[m] for m in recent_moves[-max_recent_moves:]]
            lines.append(f"Recent moves: {', '.join(readable_history)}.")

    lines.append("")
    lines.append("Your move:")
    return "\n".join(lines)


def extract_move(response: str) -> Optional[str]:
    cleaned = response.strip().lower()

    if cleaned in WORD_TO_DIR:
        return WORD_TO_DIR[cleaned]

    for word in ["north", "east", "south", "west"]:
        if re.search(rf"\b{word}\b", cleaned):
            return WORD_TO_DIR[word]

    for letter in ["n", "e", "s", "w"]:
        if re.search(rf"\b{letter}\b", cleaned):
            return WORD_TO_DIR[letter]

    return None


def run_condition_b_trial(
    maze: Maze,
    policy_fn: Callable[[str], str],
    max_steps: Optional[int] = None,
    max_recent_moves: int = 12,
    stop_on_invalid: bool = False,
) -> dict:
    """
    Runs one no-coordinate local-view maze trial.

    policy_fn:
        A function that takes a prompt string and returns the model's response string.
    """

    if max_steps is None:
        max_steps = 6 * maze.size * maze.size

    coord = maze.start
    visited = {coord}
    recent_moves: List[str] = []

    invalid_moves = 0
    revisits = 0

    trajectory = [coord]
    prompts = []
    responses = []
    parsed_moves = []
    legality = []

    shortest = shortest_distance_from(maze, maze.start, maze.end)

    for step_index in range(max_steps):
        if coord == maze.end:
            actual_steps = len(recent_moves)
            return {
                "maze": maze.name,
                "size": maze.size,
                "solved": True,
                "steps": actual_steps,
                "shortest_path_length": shortest,
                "efficiency": shortest / actual_steps if actual_steps else 1.0,
                "invalid_moves": invalid_moves,
                "revisits": revisits,
                "unique_cells_visited": len(visited),
                "coverage": len(visited) / (maze.size * maze.size),
                "trajectory": trajectory,
                "moves": recent_moves,
                "prompts": prompts,
                "responses": responses,
                "parsed_moves": parsed_moves,
                "legality": legality,
            }

        prompt = build_condition_b_prompt(
            maze=maze,
            coord=coord,
            recent_moves=recent_moves,
            max_recent_moves=max_recent_moves,
        )

        response = policy_fn(prompt)
        move = extract_move(response)

        prompts.append(prompt)
        responses.append(response)
        parsed_moves.append(move)

        if move is None:
            invalid_moves += 1
            legality.append(False)
            if stop_on_invalid:
                break
            continue

        new_coord, legal, msg = apply_move(maze, coord, move)

        if not legal:
            invalid_moves += 1
            legality.append(False)
            if stop_on_invalid:
                break
            continue

        legality.append(True)
        coord = new_coord

        if coord in visited:
            revisits += 1

        visited.add(coord)
        recent_moves.append(move)
        trajectory.append(coord)

    actual_steps = len(recent_moves)

    return {
        "maze": maze.name,
        "size": maze.size,
        "solved": False,
        "steps": actual_steps,
        "shortest_path_length": shortest,
        "efficiency": 0.0,
        "invalid_moves": invalid_moves,
        "revisits": revisits,
        "unique_cells_visited": len(visited),
        "coverage": len(visited) / (maze.size * maze.size),
        "trajectory": trajectory,
        "moves": recent_moves,
        "prompts": prompts,
        "responses": responses,
        "parsed_moves": parsed_moves,
        "legality": legality,
    }


def run_condition_b_batch(
    mazes: List[Maze],
    policy_fn: Callable[[str], str],
    max_steps_by_size: Optional[Dict[int, int]] = None,
    max_recent_moves: int = 12,
) -> pd.DataFrame:
    results = []

    for maze in mazes:
        if max_steps_by_size is None:
            max_steps = 6 * maze.size * maze.size
        else:
            max_steps = max_steps_by_size.get(maze.size, 6 * maze.size * maze.size)

        result = run_condition_b_trial(
            maze=maze,
            policy_fn=policy_fn,
            max_steps=max_steps,
            max_recent_moves=max_recent_moves,
        )

        results.append(result)

    summary_rows = []

    for r in results:
        summary_rows.append({
            "maze": r["maze"],
            "size": r["size"],
            "solved": r["solved"],
            "steps": r["steps"],
            "shortest_path_length": r["shortest_path_length"],
            "efficiency": r["efficiency"],
            "invalid_moves": r["invalid_moves"],
            "revisits": r["revisits"],
            "unique_cells_visited": r["unique_cells_visited"],
            "coverage": r["coverage"],
        })

    df = pd.DataFrame(summary_rows)
    df.attrs["full_results"] = results
    return df


def summarize_batch(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("size")
        .agg(
            n=("maze", "count"),
            solved_rate=("solved", "mean"),
            mean_steps=("steps", "mean"),
            mean_shortest_path=("shortest_path_length", "mean"),
            mean_efficiency=("efficiency", "mean"),
            mean_invalid_moves=("invalid_moves", "mean"),
            mean_revisits=("revisits", "mean"),
            mean_coverage=("coverage", "mean"),
        )
        .reset_index()
    )


def print_trial_trace(result: dict, max_rows: Optional[int] = None) -> None:
    n = len(result["responses"])
    if max_rows is not None:
        n = min(n, max_rows)

    for i in range(n):
        print("=" * 80)
        print(f"STEP {i}")
        print("-" * 80)
        print(result["prompts"][i])
        print("-" * 80)
        print(f"Response: {result['responses'][i]!r}")
        print(f"Parsed move: {result['parsed_moves'][i]}")
        print(f"Legal: {result['legality'][i]}")


## Define the 15 mazes in-notebook

Five 6x6, five 7x7, and five 8x8 mazes. `#` means wall, spaces are passages, `S` is start, `E` is end.

In [2]:
MAZE_SPECS = [
    {
        "name": '6x6-1',
        "size": 6,
        "shortest_path_length": 26,
        "ascii": """# ### #######
#S  #E      #
### # ##### #
# # # #   # #
# # ### # # #
# #   # # # #
# ### # # # #
# #   # #   #
# # ### ### #
#   #   #   #
# ### ### ###
#     #     #
#############""",
    },
    {
        "name": '6x6-2',
        "size": 6,
        "shortest_path_length": 27,
        "ascii": """# ######### #
#S  #     #E#
### ### # # #
# #   # #   #
# ### # ### #
# #   #   # #
# # ##### ###
# # #   #   #
# # # # ### #
# # # #   # #
# # # ### # #
#     #     #
#############""",
    },
    {
        "name": '6x6-3',
        "size": 6,
        "shortest_path_length": 30,
        "ascii": """# ###########
#S#         #
# ### ##### #
#   #   #   #
### ### # ###
# #   # # # #
# ### # # # #
# #   # # # #
# # ### # # #
#   #   #   #
# ### ##### #
#     #E    #
####### #####""",
    },
    {
        "name": '6x6-4',
        "size": 6,
        "shortest_path_length": 24,
        "ascii": """# ###########
#S      #   #
####### # # #
#     # # # #
# ##### ### #
# #   #   # #
# # # ### # #
#   #   #   #
# ### ##### #
# # #       #
# # #########
#          E#
########### #""",
    },
    {
        "name": '6x6-5',
        "size": 6,
        "shortest_path_length": 29,
        "ascii": """# ###########
#S#   #     #
# # # # # # #
#   # # # # #
##### ### # #
#   #     # #
# ######### #
#           #
# ###########
# #     #  E 
# ### # ### #
#     #     #
#############""",
    },
    {
        "name": '7x7-1',
        "size": 7,
        "shortest_path_length": 38,
        "ascii": """# #############
#S  #         #
### # ####### #
# # #       # #
# # ####### # #
# #       # #E 
# ####### # ###
#         #   #
# ########### #
# #       #   #
# ### ### # # #
#   # # #   # #
### # # ##### #
#     #       #
###############""",
    },
    {
        "name": '7x7-2',
        "size": 7,
        "shortest_path_length": 33,
        "ascii": """# #############
#S#           #
# # ####### # #
# # #       # #
# ### ####### #
# #   #     # #
# # ### # ### #
#   #   # #   #
##### ##### ###
# #         # #
# # ######### #
#   #         #
# ####### ### #
#         #E  #
########### ###""",
    },
    {
        "name": '7x7-3',
        "size": 7,
        "shortest_path_length": 33,
        "ascii": """# ##### #######
#S    #E      #
##### # ##### #
# #   # #     #
# # ### # #####
# #   # #     #
# ### # ##### #
# #   #     # #
# # ##### ### #
#   #   # #   #
# ### # ### # #
#     # #   # #
####### # ### #
#         #   #
###############""",
    },
    {
        "name": '7x7-4',
        "size": 7,
        "shortest_path_length": 40,
        "ascii": """# #############
#S#           #
# ######### # #
#   #       # #
### # ####### #
 E#   #     # #
# ##### ### # #
#     #   #   #
# ####### #####
# #     # #   #
# # ### # # # #
#   #   #   # #
# ### ####### #
#   #         #
###############""",
    },
    {
        "name": '7x7-5',
        "size": 7,
        "shortest_path_length": 36,
        "ascii": """# ### #########
#S#  E#       #
# # ### ##### #
# #   # # #   #
# ### # # # ###
#   #   # #   #
### # ### ### #
# # #       # #
# # ####### # #
# # #   #   # #
# # # # ### # #
#   # #   # # #
# ### ### ### #
#     #       #
###############""",
    },
    {
        "name": '8x8-1',
        "size": 8,
        "shortest_path_length": 45,
        "ascii": """# # #############
#S#E  #   #     #
# ### # # # ### #
# #   # #     # #
# # ### ####### #
# #   # #   #   #
# ### # # # # # #
#   #   # # # # #
### # ### # # # #
#   #     # # # #
# ######### # ###
#     #   # #   #
##### # # ##### #
#   #   # #   # #
# # ##### # # # #
# #         #   #
#################""",
    },
    {
        "name": '8x8-2',
        "size": 8,
        "shortest_path_length": 45,
        "ascii": """# ###############
#S#             #
# ##### ####### #
# #     #     # #
# # ##### ##### #
#   # #     #   #
##### # ### # ###
 E  #   #   # # #
### # ### # # # #
#   # #   # # # #
# # # # ##### # #
# # # # #     # #
# ### # # ##### #
# #   # # #     #
# # ### # ### # #
#     #       # #
#################""",
    },
    {
        "name": '8x8-3',
        "size": 8,
        "shortest_path_length": 50,
        "ascii": """# ###############
#S        #     #
######### # # # #
#   #     # # #E 
# # # ####### ###
# # # #     #   #
# ### # ### ### #
#   # # # #   # #
# # # # # ### # #
# #   # #     # #
# ##### # ##### #
#   #   # #     #
### # ### # ### #
# # # # # #   # #
# # # # # ### # #
#     #       # #
#################""",
    },
    {
        "name": '8x8-4',
        "size": 8,
        "shortest_path_length": 39,
        "ascii": """# ######### #####
#S    #   #E    #
##### # # ##### #
#     # #       #
# ##### ####### #
#   #       #   #
### ### ### #####
# #   # # #     #
# ### # # ##### #
# #   #     #   #
# # ######### ###
# #         #   #
# ######### ### #
#         #     #
# ### ######### #
#   #           #
#################""",
    },
    {
        "name": '8x8-5',
        "size": 8,
        "shortest_path_length": 48,
        "ascii": """# ### ###########
#S  #E      #   #
### ##### # # # #
#   #   # # # # #
# ### # # ### # #
#     # # #   # #
####### # # ### #
#   #   # #   # #
# # # ### # ### #
# #   #     #   #
# ########### # #
#   #         # #
### # ######### #
#   # #       # #
# ### # ##### # #
#     #     #   #
#################""",
    },
]

MAZES = [
    parse_ascii_maze(
        spec['name'],
        spec['ascii'],
        spec['size'],
        spec['shortest_path_length'],
    )
    for spec in MAZE_SPECS
]


## Validate mazes

This checks that every maze is internally navigable from `S` to `E`, that the start has at least one internal move, and that stored shortest-path lengths are correct.

In [3]:

for maze in MAZES:
    validate_maze(maze)


6x6-1: valid | shortest path = 26 | start moves = ['east']
6x6-2: valid | shortest path = 27 | start moves = ['east']
6x6-3: valid | shortest path = 30 | start moves = ['south']
6x6-4: valid | shortest path = 24 | start moves = ['east']
6x6-5: valid | shortest path = 29 | start moves = ['south']
7x7-1: valid | shortest path = 38 | start moves = ['east']
7x7-2: valid | shortest path = 33 | start moves = ['south']
7x7-3: valid | shortest path = 33 | start moves = ['east']
7x7-4: valid | shortest path = 40 | start moves = ['south']
7x7-5: valid | shortest path = 36 | start moves = ['south']
8x8-1: valid | shortest path = 45 | start moves = ['south']
8x8-2: valid | shortest path = 45 | start moves = ['south']
8x8-3: valid | shortest path = 50 | start moves = ['east']
8x8-4: valid | shortest path = 39 | start moves = ['east']
8x8-5: valid | shortest path = 48 | start moves = ['east']


## Optional: print all ASCII mazes

In [4]:

# Print all mazes.
for spec in MAZE_SPECS:
    print("=" * 40)
    print(f"{spec['name']} | size={spec['size']} | shortest_path={spec['shortest_path_length']}")
    print(spec["ascii"])


6x6-1 | size=6 | shortest_path=26
# ### #######
#S  #E      #
### # ##### #
# # # #   # #
# # ### # # #
# #   # # # #
# ### # # # #
# #   # #   #
# # ### ### #
#   #   #   #
# ### ### ###
#     #     #
#############
6x6-2 | size=6 | shortest_path=27
# ######### #
#S  #     #E#
### ### # # #
# #   # #   #
# ### # ### #
# #   #   # #
# # ##### ###
# # #   #   #
# # # # ### #
# # # #   # #
# # # ### # #
#     #     #
#############
6x6-3 | size=6 | shortest_path=30
# ###########
#S#         #
# ### ##### #
#   #   #   #
### ### # ###
# #   # # # #
# ### # # # #
# #   # # # #
# # ### # # #
#   #   #   #
# ### ##### #
#     #E    #
####### #####
6x6-4 | size=6 | shortest_path=24
# ###########
#S      #   #
####### # # #
#     # # # #
# ##### ### #
# #   #   # #
# # # ### # #
#   #   #   #
# ### ##### #
# # #       #
# # #########
#          E#
########### #
6x6-5 | size=6 | shortest_path=29
# ###########
#S#   #     #
# # # # # # #
#   # # # # #
##### ### # #
#   #     # #
# ######### #
#   

## Example: first prompt shown to the model

No coordinates. No map. Just local affordances.

In [5]:

def parse_available_moves_from_prompt(prompt: str) -> List[str]:
    """Extract available move words from a Condition B prompt."""
    match = re.search(r"Available moves: ([^.]+)\.", prompt)
    if not match:
        return ["north", "east", "south", "west"]

    return [m.strip() for m in match.group(1).split(",")]


def parse_entered_from_from_prompt(prompt: str) -> Optional[str]:
    """
    Extract the direction of the cell the agent came from.

    If the prompt says "You entered this cell from the north",
    then moving north would immediately backtrack.
    """
    match = re.search(r"You entered this cell from the (north|east|south|west)\.", prompt)
    if not match:
        return None

    return match.group(1)


def make_seeded_random_local_policy(seed: int = 42):
    """
    Random local baseline.

    It is intentionally dumb:
    - reads only the current prompt
    - extracts the listed available moves
    - picks one randomly
    """
    rng = random.Random(seed)

    def policy(prompt: str) -> str:
        moves = parse_available_moves_from_prompt(prompt)
        return rng.choice(moves)

    return policy


def make_seeded_nonbacktracking_policy(seed: int = 42):
    """
    Slightly-better-than-random baseline.

    It still sees ONLY the prompt text. It does not know the hidden maze,
    coordinates, or visited cells.

    Rule:
    - Extract available moves.
    - Extract "You entered this cell from the X."
    - If there is a move that is NOT the immediate backtrack direction, prefer that.
    - If forced, backtrack.
    - Break ties randomly.

    This is a very small amount of embodied common sense:
    don't immediately undo your last move unless you hit a dead end.
    """
    rng = random.Random(seed)

    def policy(prompt: str) -> str:
        moves = parse_available_moves_from_prompt(prompt)
        came_from = parse_entered_from_from_prompt(prompt)

        if came_from is None:
            return rng.choice(moves)

        non_backtracking_moves = [m for m in moves if m != came_from]

        if non_backtracking_moves:
            return rng.choice(non_backtracking_moves)

        return rng.choice(moves)

    return policy


random_policy = make_seeded_random_local_policy(seed=7)
nonbacktracking_policy = make_seeded_nonbacktracking_policy(seed=7)

# Show the first local-view prompt.
example_maze = MAZES[0]
first_prompt = build_condition_b_prompt(example_maze, example_maze.start, recent_moves=[])
print(first_prompt)


You are navigating a maze.
You cannot see the full maze.
You only know which directions are open from your current cell.
Your goal is to reach the end cell.
You will be told when you are standing on the end cell.
Reply with exactly one move: north, east, south, or west.
Do not explain your reasoning.

This is the start cell.
Available moves: east.

Your move:


## Example: run one no-API trial

This uses a seeded random local policy so the notebook runs without any API key.

In [6]:

# Run one example trial with the random baseline.
random_policy = make_seeded_random_local_policy(seed=7)
example_result = run_condition_b_trial(MAZES[0], random_policy, max_steps=6 * MAZES[0].size * MAZES[0].size)

print({
    "maze": example_result["maze"],
    "solved": example_result["solved"],
    "steps": example_result["steps"],
    "shortest_path_length": example_result["shortest_path_length"],
    "invalid_moves": example_result["invalid_moves"],
    "revisits": example_result["revisits"],
    "coverage": round(example_result["coverage"], 3),
})

print("\nFirst 5 moves:", [DIR_WORDS[m] for m in example_result["moves"][:5]])


{'maze': '6x6-1', 'solved': False, 'steps': 216, 'shortest_path_length': 26, 'invalid_moves': 0, 'revisits': 211, 'coverage': 0.167}

First 5 moves: ['east', 'south', 'south', 'north', 'north']


## Inspect a trial trace

In [7]:

# Inspect the first few prompt/response steps from the example trial.
print_trial_trace(example_result, max_rows=3)


STEP 0
--------------------------------------------------------------------------------
You are navigating a maze.
You cannot see the full maze.
You only know which directions are open from your current cell.
Your goal is to reach the end cell.
You will be told when you are standing on the end cell.
Reply with exactly one move: north, east, south, or west.
Do not explain your reasoning.

This is the start cell.
Available moves: east.

Your move:
--------------------------------------------------------------------------------
Response: 'east'
Parsed move: E
Legal: True
STEP 1
--------------------------------------------------------------------------------
You are navigating a maze.
You cannot see the full maze.
You only know which directions are open from your current cell.
Your goal is to reach the end cell.
You will be told when you are standing on the end cell.
Reply with exactly one move: north, east, south, or west.
Do not explain your reasoning.

Available moves: south, west.
You 

## Run all 15 mazes with the random baseline

In [8]:

# Run all 15 mazes with the no-API random baseline.
random_policy = make_seeded_random_local_policy(seed=123)
df_random = run_condition_b_batch(MAZES, random_policy)

display(df_random)
display(summarize_batch(df_random))


,maze,size,solved,steps,shortest_path_length,efficiency,invalid_moves,revisits,unique_cells_visited,coverage
0,6x6-1,6,False,216,26,0.0,0,192,25,0.694444
1,6x6-2,6,False,216,27,0.0,0,192,25,0.694444
2,6x6-3,6,False,216,30,0.0,0,194,23,0.638889
3,6x6-4,6,False,216,24,0.0,0,192,25,0.694444
4,6x6-5,6,False,216,29,0.0,0,197,20,0.555556
5,7x7-1,7,False,294,38,0.0,0,273,22,0.448980
6,7x7-2,7,False,294,33,0.0,0,273,22,0.448980
7,7x7-3,7,False,294,33,0.0,0,267,28,0.571429
8,7x7-4,7,False,294,40,0.0,0,268,27,0.551020
9,7x7-5,7,False,294,36,0.0,0,278,17,0.346939


,size,n,solved_rate,mean_steps,mean_shortest_path,mean_efficiency,mean_invalid_moves,mean_revisits,mean_coverage
0,6,5,0.0,216.0,27.2,0.0,0.0,193.4,0.655556
1,7,5,0.0,294.0,36.0,0.0,0.0,271.8,0.473469
2,8,5,0.0,384.0,45.4,0.0,0.0,361.4,0.368750


## Run all 15 mazes with a slightly-better-than-random baseline

This baseline still sees **only the same Condition B prompt** as the LLM would.

It uses one tiny rule: **avoid immediately backtracking into the cell you just came from unless forced**.  
So if it entered from the north and has options `north, east`, it prefers `east`.  
If the only option is `north`, it backtracks.

This is not agentic, not map-aware, and not secretly using coordinates. It is just a tiny embodied-rat heuristic.

In [9]:
# Run all 15 mazes with the no-API non-backtracking baseline.
nonbacktracking_policy = make_seeded_nonbacktracking_policy(seed=123)
df_nonbacktracking = run_condition_b_batch(MAZES, nonbacktracking_policy)

df_nonbacktracking

,maze,size,solved,steps,shortest_path_length,efficiency,invalid_moves,revisits,unique_cells_visited,coverage
0,6x6-1,6,True,34,26,0.764706,0,4,31,0.861111
1,6x6-2,6,True,29,27,0.931034,0,1,29,0.805556
2,6x6-3,6,True,34,30,0.882353,0,2,33,0.916667
3,6x6-4,6,True,42,24,0.571429,0,9,34,0.944444
4,6x6-5,6,True,59,29,0.491525,0,27,33,0.916667
5,7x7-1,7,True,76,38,0.500000,0,31,46,0.938776
6,7x7-2,7,True,41,33,0.804878,0,4,38,0.775510
7,7x7-3,7,True,39,33,0.846154,0,3,37,0.755102
8,7x7-4,7,True,178,40,0.224719,0,130,49,1.000000
9,7x7-5,7,True,120,36,0.300000,0,73,48,0.979592


## Compare random vs. non-backtracking baselines

In [10]:
comparison = (
    pd.concat(
        [
            df_random.assign(policy="random"),
            df_nonbacktracking.assign(policy="non_backtracking"),
        ],
        ignore_index=True,
    )
)

comparison_summary = (
    comparison
    .groupby(["policy", "size"])
    .agg(
        n=("maze", "count"),
        solved_rate=("solved", "mean"),
        mean_steps=("steps", "mean"),
        mean_shortest_path=("shortest_path_length", "mean"),
        mean_efficiency=("efficiency", "mean"),
        mean_invalid_moves=("invalid_moves", "mean"),
        mean_revisits=("revisits", "mean"),
        mean_coverage=("coverage", "mean"),
    )
    .reset_index()
)

comparison_summary

,policy,size,n,solved_rate,mean_steps,mean_shortest_path,mean_efficiency,mean_invalid_moves,mean_revisits,mean_coverage
0,non_backtracking,6,5,1.0,39.6,27.2,0.728209,0.0,8.6,0.888889
1,non_backtracking,7,5,1.0,90.8,36.0,0.535150,0.0,48.2,0.889796
2,non_backtracking,8,5,0.4,261.2,45.4,0.235194,0.0,208.8,0.834375
3,random,6,5,0.0,216.0,27.2,0.000000,0.0,193.4,0.655556
4,random,7,5,0.0,294.0,36.0,0.000000,0.0,271.8,0.473469
5,random,8,5,0.0,384.0,45.4,0.000000,0.0,361.4,0.368750


## Example trace from the non-backtracking baseline

In [11]:
# Inspect the first few prompt/response steps from the non-backtracking baseline.
nb_example_result = df_nonbacktracking.attrs["full_results"][0]
print_trial_trace(nb_example_result, max_rows=5)

STEP 0
--------------------------------------------------------------------------------
You are navigating a maze.
You cannot see the full maze.
You only know which directions are open from your current cell.
Your goal is to reach the end cell.
You will be told when you are standing on the end cell.
Reply with exactly one move: north, east, south, or west.
Do not explain your reasoning.

This is the start cell.
Available moves: east.

Your move:
--------------------------------------------------------------------------------
Response: 'east'
Parsed move: E
Legal: True
STEP 1
--------------------------------------------------------------------------------
You are navigating a maze.
You cannot see the full maze.
You only know which directions are open from your current cell.
Your goal is to reach the end cell.
You will be told when you are standing on the end cell.
Reply with exactly one move: north, east, south, or west.
Do not explain your reasoning.

Available moves: south, west.
You 

## Condition B + directional smell

This version keeps the same no-coordinate local-view setup, but gives the rat a crude
"smell" direction pointing as-the-crow-flies toward the end cell.

Important: this smell is **not path-aware**. It is just the straight-line direction
from the current hidden cell to the hidden end cell, quantized to:

`N, S, E, W, NE, NW, SE, SW`

So the smell can be helpful in open space, but actively misleading when the maze
requires moving away from the end to route around walls.

In [12]:
SMELL_DIR_TO_WORDS = {
    "N": "north",
    "E": "east",
    "S": "south",
    "W": "west",
    "NE": "northeast",
    "NW": "northwest",
    "SE": "southeast",
    "SW": "southwest",
}

WORD_TO_SMELL_DIR = {
    "n": "N",
    "north": "N",
    "e": "E",
    "east": "E",
    "s": "S",
    "south": "S",
    "w": "W",
    "west": "W",
    "ne": "NE",
    "northeast": "NE",
    "nw": "NW",
    "northwest": "NW",
    "se": "SE",
    "southeast": "SE",
    "sw": "SW",
    "southwest": "SW",
}


def smell_direction_to_end(maze: Maze, coord: Coord) -> Optional[str]:
    """
    Returns the as-the-crow-flies direction from coord to the maze end.

    Uses hidden coordinates to generate the observation, but the rat only sees
    the resulting direction label.

    Rows increase downward, so:
    - end row < current row means N
    - end row > current row means S
    - end col < current col means W
    - end col > current col means E
    """

    if coord == maze.end:
        return None

    r, c = coord
    er, ec = maze.end

    vertical = ""
    horizontal = ""

    if er < r:
        vertical = "N"
    elif er > r:
        vertical = "S"

    if ec < c:
        horizontal = "W"
    elif ec > c:
        horizontal = "E"

    # Canonical order: vertical then horizontal, e.g. NE, SW.
    return vertical + horizontal


def build_condition_b_smell_prompt(
    maze: Maze,
    coord: Coord,
    recent_moves: List[str],
    max_recent_moves: int = 12,
) -> str:
    """
    Condition B + smell:
    - No coordinates
    - No overhead map
    - No visited markers
    - Show local open directions
    - Show whether this is the start or end cell
    - Show the direction from which the agent entered the current cell
    - Show crude as-the-crow-flies smell direction toward the end
    - Allow backtracking
    """

    lines = []

    lines.append("You are navigating a maze.")
    lines.append("You cannot see the full maze.")
    lines.append("You only know which directions are open from your current cell.")
    lines.append("Your goal is to reach the end cell.")
    lines.append("You will be told when you are standing on the end cell.")
    lines.append("You also have a crude smell sense.")
    lines.append("The smell direction points as-the-crow-flies toward the end, not necessarily along the correct path.")
    lines.append("Reply with exactly one move: north, east, south, or west.")
    lines.append("Do not explain your reasoning.")
    lines.append("")

    if coord == maze.start:
        lines.append("This is the start cell.")

    if coord == maze.end:
        lines.append("This is the end cell. The maze is solved.")
        return "\n".join(lines)

    moves = legal_moves(maze, coord)
    readable_moves = [DIR_WORDS[m] for m in moves]
    smell_dir = smell_direction_to_end(maze, coord)

    lines.append(f"Available moves: {', '.join(readable_moves)}.")
    lines.append(f"Smell direction: {smell_dir}.")

    if recent_moves:
        previous_move = recent_moves[-1]
        came_from = OPPOSITE[previous_move]
        lines.append(f"You entered this cell from the {DIR_WORDS[came_from]}.")

        if max_recent_moves:
            readable_history = [DIR_WORDS[m] for m in recent_moves[-max_recent_moves:]]
            lines.append(f"Recent moves: {', '.join(readable_history)}.")

    lines.append("")
    lines.append("Your move:")

    return "\n".join(lines)


def parse_smell_direction_from_prompt(prompt: str) -> Optional[str]:
    """Extract smell direction like N, SE, NW, etc. from a smell prompt."""
    match = re.search(r"Smell direction: (N|S|E|W|NE|NW|SE|SW)\.", prompt)
    if match:
        return match.group(1)

    # More permissive fallback, in case you later switch to words.
    match = re.search(
        r"Smell direction: (north|south|east|west|northeast|northwest|southeast|southwest)\.",
        prompt,
        flags=re.IGNORECASE,
    )
    if match:
        return WORD_TO_SMELL_DIR[match.group(1).lower()]

    return None


def make_seeded_smell_greedy_nonbacktracking_policy(seed: int = 42):
    """
    Slightly smarter smell baseline.

    It still sees ONLY the prompt text. It does not know the hidden maze,
    coordinates, visited cells, or true path.

    Rule:
    - Extract available moves.
    - Extract immediate backtrack direction.
    - Extract smell direction.
    - Avoid immediate backtracking unless forced.
    - Prefer available moves that align with one component of the smell.
      Example: smell SE prefers south or east.
    - Break ties randomly.

    This can be better than non-backtracking, but it can also be baited by mazes
    that require moving away from the end.
    """
    rng = random.Random(seed)

    def policy(prompt: str) -> str:
        moves = parse_available_moves_from_prompt(prompt)
        came_from = parse_entered_from_from_prompt(prompt)
        smell_dir = parse_smell_direction_from_prompt(prompt)

        candidate_moves = moves[:]

        if came_from is not None:
            non_backtracking_moves = [m for m in candidate_moves if m != came_from]
            if non_backtracking_moves:
                candidate_moves = non_backtracking_moves

        if smell_dir:
            smell_components = set(smell_dir)
            scored = []

            for move in candidate_moves:
                move_letter = WORD_TO_DIR[move]
                score = 1 if move_letter in smell_components else 0
                scored.append((score, move))

            best_score = max(score for score, move in scored)
            best_moves = [move for score, move in scored if score == best_score]

            return rng.choice(best_moves)

        return rng.choice(candidate_moves)

    return policy


def run_condition_b_smell_trial(
    maze: Maze,
    policy_fn: Callable[[str], str],
    max_steps: Optional[int] = None,
    max_recent_moves: int = 12,
    stop_on_invalid: bool = False,
) -> dict:
    """
    Runs one no-coordinate local-view maze trial with smell.

    The LLM receives:
    - local open directions
    - came-from direction
    - recent moves
    - crude as-the-crow-flies smell direction toward the end
    """

    if max_steps is None:
        max_steps = 6 * maze.size * maze.size

    coord = maze.start
    visited = {coord}
    recent_moves: List[str] = []

    invalid_moves = 0
    revisits = 0

    trajectory = [coord]
    prompts = []
    responses = []
    parsed_moves = []
    legality = []

    shortest = shortest_distance_from(maze, maze.start, maze.end)

    for step_index in range(max_steps):
        if coord == maze.end:
            actual_steps = len(recent_moves)
            return {
                "maze": maze.name,
                "size": maze.size,
                "solved": True,
                "steps": actual_steps,
                "shortest_path_length": shortest,
                "efficiency": shortest / actual_steps if actual_steps else 1.0,
                "invalid_moves": invalid_moves,
                "revisits": revisits,
                "unique_cells_visited": len(visited),
                "coverage": len(visited) / (maze.size * maze.size),
                "trajectory": trajectory,
                "moves": recent_moves,
                "prompts": prompts,
                "responses": responses,
                "parsed_moves": parsed_moves,
                "legality": legality,
            }

        prompt = build_condition_b_smell_prompt(
            maze=maze,
            coord=coord,
            recent_moves=recent_moves,
            max_recent_moves=max_recent_moves,
        )

        response = policy_fn(prompt)
        move = extract_move(response)

        prompts.append(prompt)
        responses.append(response)
        parsed_moves.append(move)

        if move is None:
            invalid_moves += 1
            legality.append(False)

            if stop_on_invalid:
                break

            continue

        new_coord, legal, msg = apply_move(maze, coord, move)

        if not legal:
            invalid_moves += 1
            legality.append(False)

            if stop_on_invalid:
                break

            continue

        legality.append(True)

        coord = new_coord

        if coord in visited:
            revisits += 1

        visited.add(coord)
        recent_moves.append(move)
        trajectory.append(coord)

    actual_steps = len(recent_moves)

    return {
        "maze": maze.name,
        "size": maze.size,
        "solved": False,
        "steps": actual_steps,
        "shortest_path_length": shortest,
        "efficiency": 0.0,
        "invalid_moves": invalid_moves,
        "revisits": revisits,
        "unique_cells_visited": len(visited),
        "coverage": len(visited) / (maze.size * maze.size),
        "trajectory": trajectory,
        "moves": recent_moves,
        "prompts": prompts,
        "responses": responses,
        "parsed_moves": parsed_moves,
        "legality": legality,
    }


def run_condition_b_smell_batch(
    mazes: List[Maze],
    policy_fn: Callable[[str], str],
    max_steps_by_size: Optional[Dict[int, int]] = None,
    max_recent_moves: int = 12,
) -> pd.DataFrame:
    results = []

    for maze in mazes:
        if max_steps_by_size is None:
            max_steps = 6 * maze.size * maze.size
        else:
            max_steps = max_steps_by_size.get(maze.size, 6 * maze.size * maze.size)

        result = run_condition_b_smell_trial(
            maze=maze,
            policy_fn=policy_fn,
            max_steps=max_steps,
            max_recent_moves=max_recent_moves,
        )

        results.append(result)

    summary_rows = []

    for r in results:
        summary_rows.append({
            "maze": r["maze"],
            "size": r["size"],
            "solved": r["solved"],
            "steps": r["steps"],
            "shortest_path_length": r["shortest_path_length"],
            "efficiency": r["efficiency"],
            "invalid_moves": r["invalid_moves"],
            "revisits": r["revisits"],
            "unique_cells_visited": r["unique_cells_visited"],
            "coverage": r["coverage"],
        })

    df = pd.DataFrame(summary_rows)
    df.attrs["full_results"] = results
    return df


# Example smell prompt.
example_maze = MAZES[0]
first_smell_prompt = build_condition_b_smell_prompt(example_maze, example_maze.start, recent_moves=[])
print(first_smell_prompt)

You are navigating a maze.
You cannot see the full maze.
You only know which directions are open from your current cell.
Your goal is to reach the end cell.
You will be told when you are standing on the end cell.
You also have a crude smell sense.
The smell direction points as-the-crow-flies toward the end, not necessarily along the correct path.
Reply with exactly one move: north, east, south, or west.
Do not explain your reasoning.

This is the start cell.
Available moves: east.
Smell direction: E.

Your move:


## Run smell baseline and compare against no-smell baselines

This compares the previous random/non-backtracking policies against a smell-aware
non-backtracking policy. The smell-aware policy is still intentionally crude: it
does not know the maze, only the prompt text.

In [13]:
smell_greedy_policy = make_seeded_smell_greedy_nonbacktracking_policy(seed=123)

df_smell = run_condition_b_smell_batch(MAZES, smell_greedy_policy)

comparison_with_smell = (
    pd.concat(
        [
            df_random.assign(policy="random"),
            df_nonbacktracking.assign(policy="non_backtracking"),
            df_smell.assign(policy="smell_greedy_nonbacktracking"),
        ],
        ignore_index=True,
    )
    .groupby(["policy", "size"])
    .agg(
        n=("maze", "count"),
        solved_rate=("solved", "mean"),
        mean_steps=("steps", "mean"),
        mean_efficiency=("efficiency", "mean"),
        mean_coverage=("coverage", "mean"),
        mean_revisits=("revisits", "mean"),
    )
    .reset_index()
    .sort_values(["size", "policy"])
)

comparison_with_smell

,policy,size,n,solved_rate,mean_steps,mean_efficiency,mean_coverage,mean_revisits
0,non_backtracking,6,5,1.0,39.6,0.728209,0.888889,8.6
3,random,6,5,0.0,216.0,0.000000,0.655556,193.4
6,smell_greedy_nonbacktracking,6,5,0.6,103.6,0.600000,0.672222,80.4
1,non_backtracking,7,5,1.0,90.8,0.535150,0.889796,48.2
4,random,7,5,0.0,294.0,0.000000,0.473469,271.8
7,smell_greedy_nonbacktracking,7,5,0.2,254.6,0.068041,0.371429,237.4
2,non_backtracking,8,5,0.4,261.2,0.235194,0.834375,208.8
5,random,8,5,0.0,384.0,0.000000,0.368750,361.4
8,smell_greedy_nonbacktracking,8,5,0.4,247.8,0.400000,0.434375,221.0


## Example trace from the smell condition

In [14]:
smell_result = df_smell.attrs["full_results"][0]
print(
    {
        "maze": smell_result["maze"],
        "solved": smell_result["solved"],
        "steps": smell_result["steps"],
        "shortest_path_length": smell_result["shortest_path_length"],
        "invalid_moves": smell_result["invalid_moves"],
        "revisits": smell_result["revisits"],
        "coverage": round(smell_result["coverage"], 3),
    }
)

print("\nFirst 5 moves:", [DIR_WORDS[m] for m in smell_result["moves"][:5]])

print("\nFirst smell prompt:")
print(smell_result["prompts"][0])

{'maze': '6x6-1', 'solved': False, 'steps': 216, 'shortest_path_length': 26, 'invalid_moves': 0, 'revisits': 205, 'coverage': 0.333}

First 5 moves: ['east', 'south', 'south', 'east', 'south']

First smell prompt:
You are navigating a maze.
You cannot see the full maze.
You only know which directions are open from your current cell.
Your goal is to reach the end cell.
You will be told when you are standing on the end cell.
You also have a crude smell sense.
The smell direction points as-the-crow-flies toward the end, not necessarily along the correct path.
Reply with exactly one move: north, east, south, or west.
Do not explain your reasoning.

This is the start cell.
Available moves: east.
Smell direction: E.

Your move:


## Playwright model runs — Condition B + Smell

The cells below drive LibreChat via a persistent Playwright browser session.
Each model runs all 15 mazes using the smell prompt.
Results are saved as per-maze CSVs in `results_smell/`.

### LibreChat system prompt

Before running each model, set the following as the system prompt in its LibreChat preset:

```
You are navigating a maze.
You cannot see the full maze.
You only know which directions are open from your current cell.
Your goal is to reach the end cell.
You will be told when you are standing on the end cell.
You also have a crude smell sense.
The smell direction points as-the-crow-flies toward the end, not necessarily along the correct path.
Reply with exactly one move: north, east, south, or west.
Do not explain your reasoning.
```

In [15]:
import subprocess, sys, time
from pathlib import Path

try:
    from playwright.sync_api import sync_playwright
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "playwright"], check=True)
    subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
    from playwright.sync_api import sync_playwright

# ── Configuration ─────────────────────────────────────────────────────────
LIBRECHAT_URL      = "https://pantherai.chapman.edu"  # ← fill in
PLAYWRIGHT_SESSION = str(Path.home() / ".playwright_librechat")

# Model names exactly as shown in LibreChat's model picker.
# Adjust these strings if your deployment shows them differently.
MODELS = {
    "gpt_5_2":       "GPT 5.2",
    "gemini_pro":    "Gemini 3 Pro Preview",
    "claude_sonnet": "Claude Sonnet 4.6",
    "deepseek_v3":   "DeepSeek V3",
}

ALL_MAZES     = MAZES      # all 15 mazes

print("All models will run all 15 mazes.")


All models will run all 15 mazes.


In [16]:
class LibreChatPlayer:
    """
    Drives LibreChat in a persistent Playwright browser session.

    Session cookies are stored in PLAYWRIGHT_SESSION so login persists across
    kernel restarts. On first use, log in manually in the opened browser window.
    """

    _INPUT_SELS = [
        'textarea[data-testid="text-input"]',
        '#prompt-textarea',
        'form textarea',
        'textarea',
    ]
    _RESPONSE_SELS = [
        '[data-message-author-role="assistant"]',
        '[data-message-author-role="assistant"] .markdown',
        '.agent-turn .markdown',
        '.markdown',
    ]
    _STOP_SELS = [
        'button[data-testid="stop-button"]',
        'button[aria-label*="Stop"]',
        'button[aria-label*="stop"]',
    ]
    _MODEL_BTN_SELS = [
        'button[data-testid="model-button"]',
        'button[aria-haspopup="listbox"]',
        'button[aria-haspopup="dialog"]',
        '[class*="model-select"] button',
    ]

    def __init__(self, url: str, session_dir: str, headless: bool = False):
        self.url = url.rstrip("/")
        self.session_dir = session_dir
        self.headless = headless
        self._pw = self._ctx = self._page = None

    def start(self):
        """Launch browser and navigate to LibreChat. Returns self."""
        self._pw = sync_playwright().start()
        self._ctx = self._pw.chromium.launch_persistent_context(
            user_data_dir=self.session_dir,
            headless=self.headless,
            slow_mo=30,
        )
        self._page = self._ctx.pages[0] if self._ctx.pages else self._ctx.new_page()
        self._page.goto(self.url, wait_until="domcontentloaded", timeout=30_000)
        print(f"Browser open → {self._page.url}")
        print("If you see a login screen, log in now, then re-run this cell.")
        return self

    def stop(self):
        """Close the browser and release Playwright."""
        if self._ctx: self._ctx.close()
        if self._pw: self._pw.stop()
        self._pw = self._ctx = self._page = None

    def new_conversation(self):
        """Navigate to the root URL to start a fresh conversation."""
        self._page.goto(self.url, wait_until="domcontentloaded", timeout=30_000)
        time.sleep(0.5)

    def select_model(self, model_display_name: str):
        """
        Select a model in LibreChat's model picker using a manual prompt.
        """
        print(f"  Select '{model_display_name}' manually.")
        input("  Press Enter when done...")
        return

    # ── private helpers ────────────────────────────────────────────────────────

    def _input(self):
        for sel in self._INPUT_SELS:
            loc = self._page.locator(sel).first
            try:
                loc.wait_for(state="visible", timeout=3_000)
                return loc
            except Exception:
                continue
        raise RuntimeError(
            "Could not find the LibreChat message input.\n"
            "Check LIBRECHAT_URL and confirm you are logged in."
        )

    def _scroll_to_bottom(self):
        try:
            self._page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        except Exception:
            pass

    def _last_response(self) -> str:
        for sel in self._RESPONSE_SELS:
            try:
                loc = self._page.locator(sel)
                if loc.count() > 0:
                    text = loc.last.inner_text().strip()
                    # Strip DeepSeek R1 thinking block if present
                    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
                    return text
            except Exception:
                continue
        return ""

    def _last_response_debug(self) -> tuple:
        """Returns (text, matched_selector, element_count) for diagnostics."""
        for sel in self._RESPONSE_SELS:
            try:
                loc = self._page.locator(sel)
                n = loc.count()
                if n > 0:
                    text = loc.last.inner_text().strip()
                    return text, sel, n
            except Exception as e:
                continue
        return "", None, 0

    def _streaming(self) -> bool:
        for sel in self._STOP_SELS:
            try:
                if self._page.locator(sel).first.is_visible():
                    return True
            except Exception:
                pass
        return False

    def _fingerprint(self) -> str:
        """Fingerprint of all visible response elements using innerText (not
        textContent, which can be empty when content is CSS-rendered)."""
        for sel in self._RESPONSE_SELS:
            try:
                result = self._page.locator(sel).evaluate_all(
                    "els => els.map(e => e.innerText.trim()).join('|||')"
                )
                if result:
                    return result
            except Exception:
                continue
        return ""

    def send_and_receive(self, prompt: str, timeout: int = 180) -> str:
        """Send a message, wait for the full response, and return the text."""
        inp = self._input()
        inp.click()
        inp.fill(prompt)

        pre_fp = self._fingerprint()

        inp.press("Enter")
        time.sleep(0.1)
        self._scroll_to_bottom()

        fp_changed = False
        prev_text = ""
        stable = 0
        deadline = time.time() + timeout
        last_debug_print = time.time()

        while time.time() < deadline:
            time.sleep(0.1)
            self._scroll_to_bottom()

            try:
                now = time.time()

                if not fp_changed:
                    cur_fp = self._fingerprint()
                    fp_changed = (cur_fp != pre_fp)

                # Periodic debug print
                if now - last_debug_print >= 5.0:
                    text, sel, n = self._last_response_debug()
                    streaming = self._streaming()
                    elapsed = round(now - (deadline - timeout), 1)
                    cur_fp2 = self._fingerprint()
                    print(
                        f"\n[debug +{elapsed}s] sel={sel!r} n={n} "
                        f"streaming={streaming} changed={fp_changed} "
                        f"pre_fp_len={len(pre_fp)} cur_fp_len={len(cur_fp2)} "
                        f"cur={repr(text[:30])}"
                    )
                    last_debug_print = now

                if not fp_changed:
                    continue

                current = self._last_response()
                if not current:
                    continue

                if self._streaming():
                    if current != prev_text:
                        stable = 0
                        prev_text = current
                    continue

                if current == prev_text:
                    stable += 1
                    if stable >= 5:
                        return current
                else:
                    stable = 0
                    prev_text = current
            except Exception:
                pass

        print(f"  Warning: response timed out after {timeout}s — using partial")
        return self._last_response()

    def make_policy(self, strip_preamble: bool = True):
        """
        Return a policy_fn for run_condition_b_trial.

        strip_preamble: if True (default), removes the static 7-line preamble before
          sending. Use this when a LibreChat preset already provides those instructions
          as a system prompt. Set False to send the full self-contained prompt.
        """
        step_n = [0]

        def policy(prompt: str) -> str:
            step_n[0] += 1
            to_send = prompt.split("\n\n", 1)[1] if strip_preamble else prompt
            print(f"    step {step_n[0]}...", end=" ", flush=True)
            text = self.send_and_receive(to_send)
            print(repr(text[:60].replace("\n", " ")))
            return text

        return policy


In [17]:
import pandas as pd
import os
from datetime import datetime

def save_position_log(
    result: dict,
    model_name: str,
    output_dir: str = ".",
    filename: str = None,
) -> str:
    """
    Save a step-by-step position log for a completed trial to CSV.

    Parameters
    ----------
    result      : dict returned by run_condition_b_trial
    model_name  : string label for the model used, e.g. "gpt-4o", "claude-3-5-sonnet"
    output_dir  : folder to write the CSV into (created if it doesn't exist)
    filename    : override the auto-generated filename if you want a specific name

    Returns
    -------
    str : path to the saved CSV file
    """

    traj  = result["trajectory"]
    moves = result["moves"]

    rows = []
    for i, pos in enumerate(traj):
        move_taken = DIR_WORDS[moves[i]] if i < len(moves) else "—"
        rows.append({
            "model":        model_name,
            "maze":         result["maze"],
            "maze_size":    result["size"],
            "solved":       result["solved"],
            "total_steps":  result["steps"],
            "shortest_path_length": result["shortest_path_length"],
            "efficiency":   round(result["efficiency"], 4),
            "step":         i,
            "row":          pos[0],
            "col":          pos[1],
            "position":     str(pos),
            "move_taken":   move_taken,
        })

    df = pd.DataFrame(rows)

    os.makedirs(output_dir, exist_ok=True)

    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        safe_model = model_name.replace("/", "-").replace(" ", "_")
        filename = f"{safe_model}__{result['maze']}__{timestamp}.csv"

    filepath = os.path.join(output_dir, filename)
    df.to_csv(filepath, index=False)

    print(f"Saved {len(rows)} rows → {filepath}")
    print(f"  Maze: {result['maze']} | Model: {model_name} | Solved: {result['solved']} | Steps: {result['steps']}")

    return filepath

In [20]:
import os

def remaining_mazes(model_name, all_mazes, results_dir="results_smell"):
    """Return mazes not yet saved in results_dir for the given model."""
    safe = model_name.replace("/", "-").replace(" ", "_")
    done = set()
    if os.path.isdir(results_dir):
        for fname in os.listdir(results_dir):
            if fname.startswith(safe + "__") and fname.endswith(".csv"):
                # filename: {model}__{maze}__{timestamp}.csv
                maze_name = fname[len(safe) + 2:].rsplit("__", 1)[0]
                done.add(maze_name)
    left = [m for m in all_mazes if m.name not in done]
    print(f"[{model_name}] {len(done)} maze(s) done, {len(left)} remaining: {[m.name for m in left]}")
    return left


### GPT 5.2

In [21]:
import threading, asyncio, sys

def _run_gpt_5_2():
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    mazes = remaining_mazes("gpt-5.2", ALL_MAZES)
    if not mazes:
        print("All mazes already done for gpt-5.2.")
        return
    player = LibreChatPlayer(LIBRECHAT_URL, PLAYWRIGHT_SESSION)
    player.start()
    try:
        player.select_model(MODELS["gpt_5_2"])
        for maze in mazes:
            print(f"\n{'='*55}\nMaze: {maze.name}\n{'='*55}")
            player.new_conversation()
            policy = player.make_policy(strip_preamble=True)
            result = run_condition_b_smell_trial(maze, policy)
            save_position_log(result, model_name="gpt-5.2", output_dir="results_smell")
            print(f"  → Solved: {result['solved']} | Steps: {result['steps']}")
    finally:
        player.stop()
        print("\nBrowser closed.")

t = threading.Thread(target=_run_gpt_5_2, daemon=True)
t.start()
t.join()


[gpt-5.2] 0 maze(s) done, 15 remaining: ['6x6-1', '6x6-2', '6x6-3', '6x6-4', '6x6-5', '7x7-1', '7x7-2', '7x7-3', '7x7-4', '7x7-5', '8x8-1', '8x8-2', '8x8-3', '8x8-4', '8x8-5']
Browser open → https://pantherai.chapman.edu/
If you see a login screen, log in now, then re-run this cell.
  Select 'GPT 5.2' manually.

Maze: 6x6-1
    step 1... 'east'
    step 2... 'south'
    step 3... 'south'
    step 4... 'east'
    step 5... 'south'
    step 6... 'north'
    step 7... 'west'
    step 8... 'north'
    step 9... 'north'
    step 10... 'west'
    step 11... 'east'
    step 12... 'south'
    step 13... 'south'
    step 14... 'east'
    step 15... 'south'
    step 16... 'west'
    step 17... 'south'
    step 18... 'west'
    step 19... 'north'
    step 20... 'north'
    step 21... 'north'
    step 22... 'south'
    step 23... 'south'
    step 24... 'south'
    step 25... 'east'
    step 26... 'north'
    step 27... 'east'
    step 28... 'north'
    step 29... 'west'
    step 30... 'north'
    

### Gemini 3 Pro Preview

In [22]:
import threading, asyncio, sys

def _run_gemini():
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    mazes = remaining_mazes("gemini-3-pro-preview", ALL_MAZES)
    if not mazes:
        print("All mazes already done for gemini-3-pro-preview.")
        return
    player = LibreChatPlayer(LIBRECHAT_URL, PLAYWRIGHT_SESSION)
    player.start()
    try:
        player.select_model(MODELS["gemini_pro"])
        for maze in mazes:
            print(f"\n{'='*55}\nMaze: {maze.name}\n{'='*55}")
            player.new_conversation()
            policy = player.make_policy(strip_preamble=True)
            result = run_condition_b_smell_trial(maze, policy)
            save_position_log(result, model_name="gemini-3-pro-preview", output_dir="results_smell")
            print(f"  → Solved: {result['solved']} | Steps: {result['steps']}")
    finally:
        player.stop()
        print("\nBrowser closed.")

t = threading.Thread(target=_run_gemini, daemon=True)
t.start()
t.join()


[gemini-3-pro-preview] 0 maze(s) done, 15 remaining: ['6x6-1', '6x6-2', '6x6-3', '6x6-4', '6x6-5', '7x7-1', '7x7-2', '7x7-3', '7x7-4', '7x7-5', '8x8-1', '8x8-2', '8x8-3', '8x8-4', '8x8-5']
Browser open → https://pantherai.chapman.edu/
If you see a login screen, log in now, then re-run this cell.
  Select 'Gemini 3 Pro Preview' manually.

Maze: 6x6-1
    step 1... 'east'
    step 2... 'south'
    step 3... 'south'
    step 4... 'east'
    step 5... 'south'
    step 6... 'west'
    step 7... 'south'
    step 8... 'west'
    step 9... 'north'
    step 10... 'north'
    step 11... 'north'
    step 12... 'south'
    step 13... 'south'
    step 14... 
[debug +5.1s] sel='.agent-turn .markdown' n=13 streaming=True changed=True pre_fp_len=97 cur_fp_len=97 cur='south'
'south'
    step 15... 
[debug +5.0s] sel='.agent-turn .markdown' n=14 streaming=True changed=True pre_fp_len=105 cur_fp_len=105 cur='south'
'south'
    step 16... 'east'
    step 17... 'east'
    step 18... 'north'
    step 19... 

### Claude Sonnet 4.6

In [23]:
import threading, asyncio, sys

def _run_claude():
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    mazes = remaining_mazes("claude-sonnet-4-6", ALL_MAZES)
    if not mazes:
        print("All mazes already done for claude-sonnet-4-6.")
        return
    player = LibreChatPlayer(LIBRECHAT_URL, PLAYWRIGHT_SESSION)
    player.start()
    try:
        player.select_model(MODELS["claude_sonnet"])
        for maze in mazes:
            print(f"\n{'='*55}\nMaze: {maze.name}\n{'='*55}")
            player.new_conversation()
            policy = player.make_policy(strip_preamble=True)
            result = run_condition_b_smell_trial(maze, policy)
            save_position_log(result, model_name="claude-sonnet-4-6", output_dir="results_smell")
            print(f"  → Solved: {result['solved']} | Steps: {result['steps']}")
    finally:
        player.stop()
        print("\nBrowser closed.")

t = threading.Thread(target=_run_claude, daemon=True)
t.start()
t.join()


[claude-sonnet-4-6] 0 maze(s) done, 15 remaining: ['6x6-1', '6x6-2', '6x6-3', '6x6-4', '6x6-5', '7x7-1', '7x7-2', '7x7-3', '7x7-4', '7x7-5', '8x8-1', '8x8-2', '8x8-3', '8x8-4', '8x8-5']
Browser open → https://pantherai.chapman.edu/
If you see a login screen, log in now, then re-run this cell.
  Select 'Claude Sonnet 4.6' manually.

Maze: 6x6-1
    step 1... 'east'
    step 2... 'south'
    step 3... 'south'
    step 4... 'east'
    step 5... 'south'
    step 6... 'north'
    step 7... 'west'
    step 8... 'north'
    step 9... 'north'
    step 10... 'west'
    step 11... 'east'
    step 12... 'south'
    step 13... 'south'
    step 14... 'east'
    step 15... 'south'
    step 16... 'west'
    step 17... 'south'
    step 18... 'west'
    step 19... 'north'
    step 20... 'north'
    step 21... 'north'
    step 22... 'south'
    step 23... 'south'
    step 24... 'south'
    step 25... 'east'
    step 26... 'north'
    step 27... 'east'
    step 28... 'north'
    step 29... 'west'
    ste

### DeepSeek V3

In [24]:
import threading, asyncio, sys

def _run_deepseek():
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    mazes = remaining_mazes("deepseek-v3", ALL_MAZES)
    if not mazes:
        print("All mazes already done for deepseek-v3.")
        return
    player = LibreChatPlayer(LIBRECHAT_URL, PLAYWRIGHT_SESSION)
    player.start()
    try:
        player.select_model(MODELS["deepseek_v3"])
        for maze in mazes:
            print(f"\n{'='*55}\nMaze: {maze.name}\n{'='*55}")
            player.new_conversation()
            policy = player.make_policy(strip_preamble=True)
            result = run_condition_b_smell_trial(maze, policy)
            save_position_log(result, model_name="deepseek-v3", output_dir="results_smell")
            print(f"  → Solved: {result['solved']} | Steps: {result['steps']}")
    finally:
        player.stop()
        print("\nBrowser closed.")

t = threading.Thread(target=_run_deepseek, daemon=True)
t.start()
t.join()


[deepseek-v3] 0 maze(s) done, 15 remaining: ['6x6-1', '6x6-2', '6x6-3', '6x6-4', '6x6-5', '7x7-1', '7x7-2', '7x7-3', '7x7-4', '7x7-5', '8x8-1', '8x8-2', '8x8-3', '8x8-4', '8x8-5']
Browser open → https://pantherai.chapman.edu/
If you see a login screen, log in now, then re-run this cell.
  Select 'DeepSeek V3' manually.

Maze: 6x6-1
    step 1... 'east'
    step 2... 'south'
    step 3... 'markdown Copy code         south'
    step 4... 'markdown Copy code         east'
    step 5... 'markdown Copy code         south'
    step 6... 'markdown Copy code         west'
    step 7... 'markdown Copy code         south'
    step 8... 'markdown Copy code         west'
    step 9... 'markdown Copy code         north'
    step 10... 'markdown Copy code         north'
    step 11... 'markdown Copy code         north'
    step 12... 'markdown Copy code         south'
    step 13... 'markdown Copy code         south'
    step 14... 'markdown Copy code         south'
    step 15... 'markdown Copy cod